# 07 - Word-completion heads on the draft model

**Generated by `scripts/build_kaggle_notebooks.py` (git=db73cf78fd9c-dirty src_sha256=3b29b035b0e9). Do not edit; change `src/lgmtp/` and rebuild.**

See the README. One run does everything:
1. Streams ~60k IITB en→hi pairs and trains three sets of K = 3 heads on the **frozen** Llama-3.2-1B, all from the same forward passes:
   - **v0** plain MTP;
   - **v1** word-guided (in-word loss + continuation head);
   - **v2** v1 with uncertainty weighting.
2. Runs the 1B + heads teacher-forced along Llama-3-8B's greedy FLORES+ dev/devtest outputs from Phase 1 (embedded in this notebook, 2009 records). The log prints how often the 1B's argmax agrees with the 8B. The local analysis checks it against the Phase-1 traces.
3. Times the 1B for 1–8 new tokens per call and the heads (full and continuation vocabulary).

**Settings:** GPU T4 (one is enough), Internet on, secret `HF_TOKEN` (for `meta-llama/Llama-3.2-1B-Instruct`; otherwise the ungated `unsloth` copy). **Run:** *Save Version → Save & Run All*. About 1 h.

**Output:** `/kaggle/working/wordheads.zip` → download and unzip into `results/wordheads/` (the `.npz` and `.pt` files go to `data/wordheads/`).
**If it dies after training:** attach this version's output as input and rerun. The saved heads are found and reused.

In [ ]:
!pip -q install -U "transformers==5.22.1" accelerate huggingface_hub datasets regex

In [ ]:
# === lgmtp source, generated by scripts/build_kaggle_notebooks.py: DO NOT EDIT ===
import os, sys, importlib
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
PKG = '/kaggle/working/lgmtp' if os.path.isdir('/kaggle/working') else os.path.abspath('lgmtp')
os.makedirs(PKG, exist_ok=True)
LGMTP_SOURCES = {
    '__init__': '"""lgmtp: linguistically guided multi-token prediction (CS769)."""\n',
    'boundaries': '"""Token bytes, Devanagari akshara units, word boundaries and per-token cut labels.\n\nExtracted from ``notebooks/legacy/test1_hindi_boundaries.ipynb`` (cell 2). The logic of the\nlegacy functions is unchanged; only type hints / docstrings were added, plus ``token_labels``.\n\nFixes bug B4 (never decode single byte-level tokens): everything works on token *bytes* and\nbyte offsets, and text is only decoded for whole sequences.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nimport unicodedata\nfrom collections import Counter\nfrom itertools import accumulate\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\n# --------------------------------------------------------------------------- #\n# token -> bytes\n# --------------------------------------------------------------------------- #\n\n\ndef _bytes_to_unicode() -> dict[int, str]:\n    """GPT-2 byte<->unicode table (used by Llama-3 and Qwen byte-level BPE)."""\n    bs = (list(range(ord("!"), ord("~") + 1))\n          + list(range(ord("¡"), ord("¬") + 1))\n          + list(range(ord("®"), ord("ÿ") + 1)))\n    cs = bs[:]\n    n = 0\n    for b in range(256):\n        if b not in bs:\n            bs.append(b)\n            cs.append(256 + n)\n            n += 1\n    return dict(zip(bs, [chr(c) for c in cs]))\n\n\nBYTE_DECODER: dict[str, int] = {v: k for k, v in _bytes_to_unicode().items()}\n_HEX_RE = re.compile(r"^<0x([0-9A-Fa-f]{2})>$")\n\n\ndef token_to_bytes(tok: str) -> bytes:\n    """Raw bytes of a vocabulary token string (as returned by ``convert_ids_to_tokens``)."""\n    m = _HEX_RE.match(tok)\n    if m:  # SentencePiece byte-fallback token\n        return bytes([int(m.group(1), 16)])\n    if all(ch in BYTE_DECODER for ch in tok):  # GPT-2 style byte-level BPE\n        return bytes(BYTE_DECODER[ch] for ch in tok)\n    return tok.replace("▁", " ").encode("utf-8")  # SentencePiece unicode piece\n\n\nclass HFTok:\n    """Thin wrapper: text -> list of token byte-strings."""\n\n    def __init__(self, name_or_tokenizer):\n        if isinstance(name_or_tokenizer, str):\n            from transformers import AutoTokenizer\n            self.name = name_or_tokenizer\n            self.tok = AutoTokenizer.from_pretrained(name_or_tokenizer)\n        else:\n            self.name = "custom"\n            self.tok = name_or_tokenizer\n\n    def to_bytes(self, text: str) -> list[bytes]:\n        ids = self.tok(text, add_special_tokens=False)["input_ids"]\n        return [token_to_bytes(t) for t in self.tok.convert_ids_to_tokens(ids)]\n\n    def ids_to_bytes(self, ids: Sequence[int]) -> list[bytes]:\n        """Byte-strings for a list of token ids (no decoding of individual tokens)."""\n        return [token_to_bytes(t) for t in self.tok.convert_ids_to_tokens(list(ids))]\n\n\nQWEN2_PRETOK_REGEX = (r"(?i:\'s|\'t|\'re|\'ve|\'m|\'ll|\'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}"\n                      r"| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+")\n\n\ndef legacy_qwen2_pretok_tokenizer(name: str = "Qwen/Qwen3.5-0.8B"):\n    """``name``\'s tokenizer.json with its split regex replaced by the older Qwen2 one (``\\\\p{L}+``,\n    no ``\\\\p{M}``), which splits before every combining mark.\n\n    Only used to explain the legacy Test 1 Qwen number (3.33 tokens/word): the legacy run loaded\n    the tokenizer this way. The model\'s real tokenizer (``[\\\\p{L}\\\\p{M}]+``) gives ~2.16.\n    """\n    from huggingface_hub import hf_hub_download\n    from tokenizers import Tokenizer\n    from transformers import PreTrainedTokenizerFast\n    j = json.load(open(hf_hub_download(name, "tokenizer.json"), encoding="utf-8"))\n    j["pre_tokenizer"]["pretokenizers"][0]["pattern"]["Regex"] = QWEN2_PRETOK_REGEX\n    return PreTrainedTokenizerFast(tokenizer_object=Tokenizer.from_str(json.dumps(j)))\n\n\n# --------------------------------------------------------------------------- #\n# akshara + word boundaries\n# --------------------------------------------------------------------------- #\n\n_C = "क-हक़-य़ॸ-ॿ"      # consonants\n_VIRAMA = "्"\n_NUKTA = "़"\n_ZW = "\u200c\u200d"                                 # ZWNJ / ZWJ\n_IVOW = "ऄ-औॠॡॲ-ॷ"     # independent vowels\n_MATRA = "ऺऻा-ौॎॏॕ-ॗॢॣ"\n_MOD = "ऀ-ः॑-॔"                  # candrabindu, anusvara, visarga, stress marks\n\n_AKSHARA_RE = re.compile(\n    rf"[{_C}]{_NUKTA}?(?:{_VIRAMA}[{_ZW}]?[{_C}]{_NUKTA}?)*(?:{_VIRAMA}[{_ZW}]?|[{_MATRA}]*)[{_MOD}]*"\n    rf"|[{_IVOW}]{_NUKTA}?[{_MATRA}]*[{_MOD}]*"\n)\n\n\ndef akshara_units(text: str) -> list[tuple[int, int]]:\n    """Return list of (start, end) char spans. Non-Devanagari chars are single units\n    with any following combining marks / ZW joiners attached."""\n    units, i, n = [], 0, len(text)\n    while i < n:\n        m = _AKSHARA_RE.match(text, i)\n        if m and m.end() > i:\n            j = m.end()\n        else:\n            j = i + 1\n            while j < n and (unicodedata.category(text[j]).startswith("M") or text[j] in "\u200c\u200d"):\n                j += 1\n        units.append((i, j))\n        i = j\n    return units\n\n\ndef is_word_boundary(text: str, c: int) -> bool:\n    """True if char offset ``c`` is a word boundary (text edge, whitespace or punctuation)."""\n    n = len(text)\n    if c <= 0 or c >= n:\n        return True\n    a, b = text[c - 1], text[c]\n    if a.isspace() or b.isspace():\n        return True\n    if unicodedata.category(a).startswith("P") or unicodedata.category(b).startswith("P"):\n        return True\n    return False\n\n\nEND, WORD, MIDWORD, MIDAKS, MIDUTF8 = "end", "word_boundary", "mid_word", "mid_akshara", "mid_utf8"\nCUT_CATEGORIES = (END, WORD, MIDWORD, MIDAKS, MIDUTF8)\nTAG = {END: "E", WORD: "W", MIDWORD: "w", MIDAKS: "a", MIDUTF8: "u"}\n\n\ndef classify(b: int, total: int, byte2char: dict[int, int], text: str, aks: set[int]) -> str:\n    """Category of a cut at byte offset ``b``.\n\n    byte2char maps byte offsets that start a char (plus ``total``) to char offsets;\n    aks is the set of char offsets that are akshara-unit boundaries.\n    """\n    if b >= total:\n        return END\n    c = byte2char.get(b)\n    if c is None:\n        return MIDUTF8\n    if c not in aks:\n        return MIDAKS\n    return WORD if is_word_boundary(text, c) else MIDWORD\n\n\ndef sentence_features(tb: Sequence[bytes]) -> dict | None:\n    """tb: list of token byte-strings. Returns None if bytes are not valid UTF-8."""\n    full = b"".join(tb)\n    try:\n        text = full.decode("utf-8")\n    except UnicodeDecodeError:\n        return None\n    byte2char, pos = {}, 0\n    for ci, ch in enumerate(text):\n        byte2char[pos] = ci\n        pos += len(ch.encode("utf-8"))\n    byte2char[pos] = len(text)\n    units = akshara_units(text)\n    aks = {0} | {e for _, e in units}\n    ends = list(accumulate(len(b) for b in tb))\n    starts = [0] + ends[:-1]\n    total = len(full)\n    return {\n        "text": text,\n        "units": units,\n        "cut": [classify(e, total, byte2char, text, aks) for e in ends],\n        "start": [classify(s, total, byte2char, text, aks) for s in starts],\n    }\n\n\n# --------------------------------------------------------------------------- #\n# per-token gold labels (analysis only; never read by policies)\n# --------------------------------------------------------------------------- #\n\n\ndef _valid_utf8_prefix_len(full: bytes) -> int:\n    """Length of the longest prefix of ``full`` that is valid UTF-8, allowing only a\n    truncated character at the very end (e.g. generation stopped at MAX_NEW)."""\n    for cut in range(len(full), max(len(full) - 4, -1), -1):\n        try:\n            full[:cut].decode("utf-8")\n            return cut\n        except UnicodeDecodeError:\n            continue\n    raise ValueError("token bytes are not valid UTF-8 (invalid sequence before the end)")\n\n\ndef token_labels(token_bytes: Sequence[bytes],\n                 special: Sequence[bool] | None = None) -> dict[str, np.ndarray]:\n    """Gold labels for each token t of a target sequence (analysis only, never read by policies).\n\n    end_cut[t]      : category of the cut AFTER token t (END/WORD/MIDWORD/MIDAKS/MIDUTF8)\n    start_cut[t]    : category of the cut BEFORE token t\n    word_initial[t] : token t starts a new word (start_cut in {WORD, END})\n    pos_in_word[t]  : 0 for word-initial tokens, then 1, 2, ...\n    word_id[t]      : index of the word the token belongs to\n    char_span[t]    : (start_char, end_char) in the decoded text, shape (n, 2)\n\n    ``special[t]`` marks special/control tokens (e.g. ``<|eot_id|>``). They are stripped before\n    labelling and get ``END`` for both cuts, ``word_initial=True``, ``pos_in_word=0``, a fresh\n    ``word_id`` and an empty char span at the end of the text. If ``special`` is None, no token is\n    treated as special.\n\n    A trailing incomplete UTF-8 character (output truncated mid-character) is tolerated: cuts at\n    its start or inside it are ``MIDUTF8`` (what follows is unknown). Invalid bytes anywhere else\n    raise ``ValueError``.\n    """\n    n = len(token_bytes)\n    special = [False] * n if special is None else [bool(s) for s in special]\n    if len(special) != n:\n        raise ValueError("special must have the same length as token_bytes")\n    keep = [t for t in range(n) if not special[t]]\n    tb = [token_bytes[t] for t in keep]\n\n    full = b"".join(tb)\n    valid = _valid_utf8_prefix_len(full)\n    text = full[:valid].decode("utf-8")\n    full_total = len(full)\n\n    byte2char, pos = {}, 0\n    char_of_byte = np.zeros(full_total + 1, dtype=np.int64)  # char containing byte b\n    for ci, ch in enumerate(text):\n        byte2char[pos] = ci\n        nb = len(ch.encode("utf-8"))\n        char_of_byte[pos:pos + nb] = ci\n        pos += nb\n    byte2char[pos] = len(text)\n    char_of_byte[pos:] = len(text)  # trailing partial char maps past the text\n    aks = {0} | {e for _, e in akshara_units(text)}\n\n    def cat(b: int) -> str:\n        if b >= full_total:\n            return END\n        if b >= valid:  # inside the trailing truncated char\n            return MIDUTF8\n        return classify(b, valid, byte2char, text, aks)\n\n    ends = list(accumulate(len(b) for b in tb))\n    starts = [0] + ends[:-1]\n\n    end_cut = np.full(n, END, dtype=object)\n    start_cut = np.full(n, END, dtype=object)\n    char_span = np.full((n, 2), len(text), dtype=np.int64)\n    for j, t in enumerate(keep):\n        s, e = starts[j], ends[j]\n        end_cut[t] = cat(e)\n        start_cut[t] = cat(s)\n        cs = int(char_of_byte[min(s, full_total)])\n        ce = int(char_of_byte[e - 1]) + 1 if e > s else cs\n        char_span[t] = (cs, min(ce, len(text)) if e <= valid else len(text))\n\n    word_initial = np.array([sc in (WORD, END) for sc in start_cut], dtype=bool)\n    pos_in_word = np.zeros(n, dtype=np.int64)\n    word_id = np.zeros(n, dtype=np.int64)\n    wid = -1\n    for t in range(n):\n        if word_initial[t] or t == 0:\n            wid += 1\n            pos_in_word[t] = 0\n        else:\n            pos_in_word[t] = pos_in_word[t - 1] + 1\n        word_id[t] = wid\n    return dict(end_cut=end_cut, start_cut=start_cut, word_initial=word_initial,\n                pos_in_word=pos_in_word, word_id=word_id, char_span=char_span)\n\n\n# --------------------------------------------------------------------------- #\n# corpus analysis\n# --------------------------------------------------------------------------- #\n\n\ndef analyze(token_bytes_per_sentence: Iterable[Sequence[bytes]], ks: Sequence[int]):\n    """Corpus statistics of token cuts and of fixed-k draft windows. Returns (tot, per_k)."""\n    tot = Counter()\n    per_k = {k: Counter() for k in ks}\n    for tb in token_bytes_per_sentence:\n        f = sentence_features(tb)\n        if f is None:\n            tot["skipped_invalid_utf8"] += 1\n            continue\n        n = len(tb)\n        if n < 2:\n            tot["skipped_short"] += 1\n            continue\n        text, cut, start = f["text"], f["cut"], f["start"]\n        tot["sentences"] += 1\n        tot["tokens"] += n\n        tot["words"] += len(text.split())\n        tot["chars"] += len(text)\n        tot["aksharas"] += len(f["units"])\n        tot["starts_mid_utf8"] += sum(1 for s in start if s == MIDUTF8)\n        tot["starts_mid_akshara"] += sum(1 for s in start if s == MIDAKS)\n        for j in range(n - 1):  # internal cuts only (a token follows)\n            tot["cuts"] += 1\n            tot["cut_" + cut[j]] += 1\n\n        wb = [c in (WORD, END) for c in cut]\n        pw = [0] + list(accumulate(1 if x else 0 for x in wb))\n        last_wb, lw = [], -1\n        for j in range(n):\n            if wb[j]:\n                lw = j\n            last_wb.append(lw)\n        next_wb, nw = [0] * n, n - 1\n        for j in range(n - 1, -1, -1):\n            if wb[j]:\n                nw = j\n            next_wb[j] = nw\n\n        for k in ks:\n            c = per_k[k]\n            for s in range(0, n - k):\n                e = s + k - 1\n                cat = cut[e]\n                c["windows"] += 1\n                c[cat] += 1\n                done = pw[e + 1] - pw[s]\n                c["completed_words"] += done\n                if done == 0:\n                    c["zero_word"] += 1\n                if cat not in (WORD, END):\n                    c["mid_n"] += 1\n                    c["fwd_sum"] += next_wb[e] - e\n                    c["back_sum"] += e - last_wb[e]\n    return tot, per_k\n\n\ndef pct(a: float, b: float) -> float:\n    return 100.0 * a / b if b else float("nan")\n\n\ndef summarize(name: str, tot: Counter, per_k: dict, ks: Sequence[int]) -> dict:\n    """Turn ``analyze`` counters into the Test 1 summary dict."""\n    S = {\n        "tokenizer": name,\n        "sentences": tot["sentences"],\n        "tokens": tot["tokens"],\n        "tokens_per_word": tot["tokens"] / max(tot["words"], 1),\n        "tokens_per_akshara": tot["tokens"] / max(tot["aksharas"], 1),\n        "tokens_per_char": tot["tokens"] / max(tot["chars"], 1),\n        "pct_tokens_start_mid_akshara": pct(tot["starts_mid_akshara"], tot["tokens"]),\n        "pct_tokens_start_mid_utf8": pct(tot["starts_mid_utf8"], tot["tokens"]),\n        "pct_cuts_word_boundary": pct(tot["cut_" + WORD] + tot["cut_" + END], tot["cuts"]),\n        "pct_cuts_mid_word": pct(tot["cut_" + MIDWORD], tot["cuts"]),\n        "pct_cuts_mid_akshara": pct(tot["cut_" + MIDAKS], tot["cuts"]),\n        "pct_cuts_mid_utf8": pct(tot["cut_" + MIDUTF8], tot["cuts"]),\n        "skipped_invalid_utf8": tot["skipped_invalid_utf8"],\n        "per_k": [],\n    }\n    for k in ks:\n        c = per_k[k]\n        w = c["windows"]\n        S["per_k"].append({\n            "k": k,\n            "windows": w,\n            "pct_end_at_word_boundary": pct(c[WORD] + c[END], w),\n            "pct_end_mid_word": pct(c[MIDWORD], w),\n            "pct_end_mid_akshara": pct(c[MIDAKS], w),\n            "pct_end_mid_utf8": pct(c[MIDUTF8], w),\n            "pct_end_anywhere_inside_word": 100.0 - pct(c[WORD] + c[END], w) if w else float("nan"),\n            "mean_words_completed": c["completed_words"] / w if w else float("nan"),\n            "pct_drafts_completing_zero_words": pct(c["zero_word"], w),\n            "mean_tokens_forward_to_boundary": c["fwd_sum"] / c["mid_n"] if c["mid_n"] else float("nan"),\n            "mean_tokens_back_to_boundary": c["back_sum"] / c["mid_n"] if c["mid_n"] else float("nan"),\n        })\n    return S\n\n\ndef print_summary(S: dict) -> None:\n    print(f"\\n=== {S[\'tokenizer\']} ===")\n    print(f"sentences={S[\'sentences\']}  tokens={S[\'tokens\']}  "\n          f"tokens/word={S[\'tokens_per_word\']:.2f}  tokens/akshara={S[\'tokens_per_akshara\']:.2f}  "\n          f"tokens/char={S[\'tokens_per_char\']:.2f}")\n    print(f"tokens that START mid-akshara: {S[\'pct_tokens_start_mid_akshara\']:.1f}%   "\n          f"mid-UTF-8: {S[\'pct_tokens_start_mid_utf8\']:.1f}%")\n    print(f"all token cuts -> word boundary {S[\'pct_cuts_word_boundary\']:.1f}% | mid-word {S[\'pct_cuts_mid_word\']:.1f}% | "\n          f"mid-akshara {S[\'pct_cuts_mid_akshara\']:.1f}% | mid-UTF-8 {S[\'pct_cuts_mid_utf8\']:.1f}%")\n    if S["skipped_invalid_utf8"]:\n        print(f"WARNING: skipped {S[\'skipped_invalid_utf8\']} sentences (token bytes not valid UTF-8)")\n    hdr = ("k", "@word", "mid-word", "mid-aks", "mid-utf8", "inside-any", "words/draft", "0-word%", "fwd", "back")\n    print("\\n" + " ".join(f"{h:>11}" for h in hdr))\n    for r in S["per_k"]:\n        print(" ".join(f"{v:>11}" for v in (\n            r["k"], f"{r[\'pct_end_at_word_boundary\']:.1f}", f"{r[\'pct_end_mid_word\']:.1f}",\n            f"{r[\'pct_end_mid_akshara\']:.1f}", f"{r[\'pct_end_mid_utf8\']:.1f}",\n            f"{r[\'pct_end_anywhere_inside_word\']:.1f}", f"{r[\'mean_words_completed\']:.2f}",\n            f"{r[\'pct_drafts_completing_zero_words\']:.1f}",\n            f"{r[\'mean_tokens_forward_to_boundary\']:.2f}", f"{r[\'mean_tokens_back_to_boundary\']:.2f}")))\n    print("(columns 2-6 are % of k-token drafts; fwd/back = avg tokens to move to the nearest word boundary "\n          "when the draft ends inside a word)")\n\n\n# --------------------------------------------------------------------------- #\n# examples for slides\n# --------------------------------------------------------------------------- #\n\n\ndef _show(b: bytes) -> str:\n    try:\n        return b.decode("utf-8").replace(" ", "␣")  # visible space\n    except UnicodeDecodeError:\n        return "<" + b.hex() + ">"\n\n\ndef render_example(tb: Sequence[bytes], k: int) -> str:\n    """Render a tokenised sentence with fixed-k draft ends tagged by cut category."""\n    f = sentence_features(tb)\n    if f is None:\n        return "(invalid utf-8)"\n    out = []\n    for i, b in enumerate(tb):\n        out.append(_show(b))\n        if i < len(tb) - 1:\n            out.append(f" ‖[{TAG[f[\'cut\'][i]]}] " if (i + 1) % k == 0 else " | ")\n    return "".join(out)\n\n\nEXAMPLE_LEGEND = ("Legend: | token boundary,  ‖ end of a fixed-k draft,  tag = what the draft ends on: "\n                  "W=word boundary, w=inside word, a=inside akshara, u=inside UTF-8 char, ␣ = space\\n")\n\n# --------------------------------------------------------------------------- #\n# data\n# --------------------------------------------------------------------------- #\n\nDEMO = [\n    "भारत एक विशाल और विविधताओं से भरा देश है।",\n    "विद्यार्थियों ने विद्यालय में सूर्योदय के समय प्रार्थना की।",\n    "मौसम विभाग ने कल भारी बारिश की चेतावनी जारी की है।",\n    "वैज्ञानिकों का मानना है कि जलवायु परिवर्तन का प्रभाव कृषि पर पड़ेगा।",\n    "उसने अपने दोस्तों को खाने पर बुलाया और सबने मिलकर खाना खाया।",\n]\n\n\ndef load_sentences(path: str, field: str) -> list[str]:\n    out = []\n    with open(path, encoding="utf-8") as fh:\n        if path.endswith(".jsonl"):\n            for line in fh:\n                if line.strip():\n                    out.append(json.loads(line)[field])\n        else:\n            out = [ln.rstrip("\\n") for ln in fh]\n    return [unicodedata.normalize("NFC", s.strip()) for s in out if s.strip()]\n\n\ndef load_flores_hindi(split: str = "devtest") -> list[str]:\n    """FLORES+ hin_Deva sentences (NFC, stripped), as used in Test 1."""\n    from datasets import load_dataset\n    ds = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split=split)\n    return [unicodedata.normalize("NFC", s.strip()) for s in ds["text"] if s.strip()]\n',
    'vocab': '"""Boolean masks over the full vocabulary, built from token *bytes* (``boundaries.token_to_bytes``).\n\nMasks (all shape ``(len(tok),)``, dtype bool):\n\n- ``starts_space``: first byte in ``b" \\\\n\\\\t"``.\n- ``starts_punct``: first decoded char has Unicode category ``P*`` (``।``, ``,``, ``.``, ``"``, ...).\n- ``starts_mark``: first decoded char has category ``M*`` (matra, virama, nukta, anusvara,\n  candrabindu, visarga, ...) or is ZWJ / ZWNJ.\n- ``starts_cont``: first byte is a UTF-8 continuation byte (0x80-0xBF): the token starts mid-character.\n  Such a token is never ``starts_mark`` / ``starts_punct``.\n- ``is_special``: special / control tokens (all registered special tokens and eos ids).\n\nA token whose bytes start with a lead byte but do not contain a complete first character\n(e.g. ``b"\\\\xe0\\\\xa4"``) has an unknown first char and is in none of space/punct/mark/cont.\nSpecial tokens are only ``is_special``.\n"""\nfrom __future__ import annotations\n\nimport unicodedata\nfrom dataclasses import dataclass, field\nfrom typing import Iterable\n\nimport numpy as np\n\nfrom .boundaries import token_to_bytes\n\nZW = "\u200c\u200d"\nMASK_NAMES = ("starts_space", "starts_punct", "starts_mark", "starts_cont", "is_special")\n\n\ndef first_char(b: bytes) -> str:\n    """First complete character of ``b`` (\'\' if none). Continuation-byte starts give \'\'."""\n    if not b or 0x80 <= b[0] <= 0xBF:\n        return ""\n    return b[:4].decode("utf-8", errors="ignore")[:1]\n\n\ndef classify_token_bytes(b: bytes) -> dict[str, bool]:\n    """Mask flags for one non-special token given its raw bytes."""\n    out = dict(starts_space=False, starts_punct=False, starts_mark=False, starts_cont=False,\n               is_special=False)\n    if not b:\n        return out\n    if b[0] in b" \\n\\t":\n        out["starts_space"] = True\n        return out\n    if 0x80 <= b[0] <= 0xBF:\n        out["starts_cont"] = True\n        return out\n    ch = first_char(b)\n    if not ch:\n        return out\n    cat = unicodedata.category(ch)\n    if cat.startswith("P"):\n        out["starts_punct"] = True\n    elif cat.startswith("M") or ch in ZW:\n        out["starts_mark"] = True\n    return out\n\n\n@dataclass\nclass VocabMasks:\n    starts_space: np.ndarray\n    starts_punct: np.ndarray\n    starts_mark: np.ndarray\n    starts_cont: np.ndarray\n    is_special: np.ndarray\n    eos_ids: list[int] = field(default_factory=list)\n\n    @property\n    def size(self) -> int:\n        return len(self.starts_space)\n\n    @property\n    def is_eos(self) -> np.ndarray:\n        m = np.zeros(self.size, dtype=bool)\n        m[[i for i in self.eos_ids if i < self.size]] = True\n        return m\n\n    @property\n    def starts_newword(self) -> np.ndarray:\n        """Tokens that end the current word if they come next: space, punctuation or eos."""\n        return self.starts_space | self.starts_punct | self.is_eos\n\n    def padded(self, name: str, size: int) -> np.ndarray:\n        """Mask padded with False to ``size`` (model logits can be wider than ``len(tok)``)."""\n        m = getattr(self, name)\n        if size < len(m):\n            raise ValueError("size smaller than vocabulary")\n        return np.concatenate([m, np.zeros(size - len(m), dtype=bool)])\n\n    def as_dict(self) -> dict[str, np.ndarray]:\n        return {k: getattr(self, k) for k in MASK_NAMES}\n\n\ndef special_ids(tok) -> set[int]:\n    """All special/control token ids of a Hugging Face tokenizer."""\n    ids = set(getattr(tok, "all_special_ids", []) or [])\n    for i, at in (getattr(tok, "added_tokens_decoder", {}) or {}).items():\n        if getattr(at, "special", False):\n            ids.add(int(i))\n    return ids\n\n\ndef build_masks(tok, eos_ids: Iterable[int] = ()) -> VocabMasks:\n    """Build all masks for a Hugging Face tokenizer (size ``len(tok)``)."""\n    n = len(tok)\n    eos_ids = sorted(set(int(i) for i in eos_ids))\n    spec = special_ids(tok) | set(eos_ids)\n    toks = tok.convert_ids_to_tokens(list(range(n)))\n    arrs = {k: np.zeros(n, dtype=bool) for k in MASK_NAMES}\n    for i, t in enumerate(toks):\n        if i in spec or t is None:\n            arrs["is_special"][i] = True\n            continue\n        for k, v in classify_token_bytes(token_to_bytes(t)).items():\n            if v:\n                arrs[k][i] = True\n    return VocabMasks(**arrs, eos_ids=eos_ids)\n\n\ndef token_bytes_list(tok, ids) -> list[bytes]:\n    """Raw bytes of each id (special tokens give their literal text bytes)."""\n    return [token_to_bytes(t) for t in tok.convert_ids_to_tokens([int(i) for i in ids])]\n',
    'traces': '"""Trace schema (one target sentence), compressed ``.npz`` save/load, gold-label attachment.\n\nIndex convention: index ``t`` refers to target token ``y[t]``; every draft feature at\n``t`` is a function of ``q_t``, the draft distribution over ``y[t]`` computed teacher-forced on\n``prompt + y[:t]``.\n\nStorage: per-position arrays of all sentences are concatenated into flat arrays and stored with\nan ``offsets`` array (sentence i occupies ``offsets[i]:offsets[i+1]``). ``meta`` is stored as a\nJSON string.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field, fields\nfrom pathlib import Path\n\nimport numpy as np\n\n# per-position 1-D arrays saved to disk (in this order)\nSCALAR_FIELDS = ("target_ids", "draft_argmax", "H", "top1", "top2", "p_target",\n                 "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont",\n                 "target_logp_of_draft_argmax")\n# per-position (n, K) arrays\nTOPK_FIELDS = ("draft_topk_ids", "draft_topk_logp", "target_topk_ids", "target_topk_logp")\nINT_FIELDS = {"target_ids", "draft_argmax", "draft_topk_ids", "target_topk_ids"}\nTOPK = 20\n\n\n@dataclass\nclass Trace:\n    """One sentence. All arrays have first dimension ``n``."""\n    id: int\n    n: int\n    target_ids: np.ndarray\n    draft_argmax: np.ndarray\n    H: np.ndarray            # draft entropy (nats)\n    top1: np.ndarray         # draft prob of its argmax\n    top2: np.ndarray         # draft prob of its 2nd-best token\n    p_target: np.ndarray     # q_t(y[t])\n    p_space: np.ndarray\n    p_punct: np.ndarray\n    p_eos: np.ndarray\n    p_newword: np.ndarray    # p_space + p_punct + p_eos\n    p_mark: np.ndarray\n    p_cont: np.ndarray\n    draft_topk_ids: np.ndarray\n    draft_topk_logp: np.ndarray\n    target_topk_ids: np.ndarray\n    target_topk_logp: np.ndarray\n    target_logp_of_draft_argmax: np.ndarray\n    match: np.ndarray = None  # derived: draft_argmax == target_ids\n    labels: dict = field(default_factory=dict, repr=False)  # gold labels (analysis only, not saved)\n\n    def __post_init__(self):\n        self.match = np.asarray(self.draft_argmax) == np.asarray(self.target_ids)\n        for f in SCALAR_FIELDS + TOPK_FIELDS:\n            a = getattr(self, f)\n            if len(a) != self.n:\n                raise ValueError(f"trace {self.id}: {f} has length {len(a)} != n={self.n}")\n\n\ndef save_traces(path, traces: list[Trace], meta: dict | None = None) -> None:\n    """Save traces as one compressed ``.npz`` (flat arrays + offsets)."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    lens = np.array([t.n for t in traces], dtype=np.int64)\n    offsets = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    out = dict(ids=np.array([t.id for t in traces], dtype=np.int64), offsets=offsets,\n               meta=np.array(json.dumps(meta or {}, ensure_ascii=False)))\n    for f in SCALAR_FIELDS + TOPK_FIELDS:\n        dt = np.int32 if f in INT_FIELDS else np.float32\n        parts = [np.asarray(getattr(t, f), dtype=dt) for t in traces]\n        if parts:\n            out[f] = np.concatenate(parts, axis=0)\n        else:\n            out[f] = np.zeros((0, TOPK) if f in TOPK_FIELDS else (0,), dtype=dt)\n    np.savez_compressed(path, **out)\n\n\ndef load_traces(path) -> tuple[list[Trace], dict]:\n    """Inverse of ``save_traces``."""\n    z = np.load(path, allow_pickle=False)\n    meta = json.loads(str(z["meta"]))\n    offsets, ids = z["offsets"], z["ids"]\n    flat = {f: z[f] for f in SCALAR_FIELDS + TOPK_FIELDS}\n    traces = []\n    for i, sid in enumerate(ids):\n        a, b = int(offsets[i]), int(offsets[i + 1])\n        traces.append(Trace(id=int(sid), n=b - a, **{f: flat[f][a:b] for f in flat}))\n    return traces, meta\n\n\ndef summary(traces: list[Trace]) -> dict:\n    """Positions, agreement rate, mean draft entropy when match / mismatch."""\n    m = np.concatenate([t.match for t in traces])\n    H = np.concatenate([t.H for t in traces])\n    return dict(sentences=len(traces), positions=int(len(m)), agreement=float(m.mean()),\n                mean_H=float(H.mean()), mean_H_match=float(H[m].mean()) if m.any() else float("nan"),\n                mean_H_mismatch=float(H[~m].mean()) if (~m).any() else float("nan"))\n\n\ndef attach_word_labels(traces: list[Trace], tok) -> None:\n    """Attach gold word/akshara labels from the *target* tokens (fixes B5). Analysis only."""\n    from .boundaries import token_labels\n    from .vocab import special_ids, token_bytes_list\n    from .boundaries import _valid_utf8_prefix_len\n    spec = special_ids(tok)\n    for t in traces:\n        tb = token_bytes_list(tok, t.target_ids)\n        sp = [int(i) in spec for i in t.target_ids]\n        t.labels.update(token_labels(tb, special=sp))\n        full = b"".join(b for b, s in zip(tb, sp) if not s)\n        t.labels["text"] = full[:_valid_utf8_prefix_len(full)].decode("utf-8")  # text the char spans refer to\n\n\ndef trace_field_names() -> list[str]:\n    return [f.name for f in fields(Trace)]\n',
    'gpu_traces': '"""Target generation, teacher-forced draft/target passes and timing (Kaggle notebook 01).\n\nPorted from the legacy Part A (``notebooks/legacy/day2-day3-llama8b-kaggle.ipynb`` cells 4-11).\nEverything here also runs on CPU (``scripts/smoke_cpu.py``) with a small model in fp32.\n\nOutputs (in ``out_dir``) for one split:\n  target_outputs_{split}.jsonl   id, src, ref, prompt_ids, out_ids, out_text   (append + resume)\n  _draft_tf_{split}.pkl          partial draft teacher-forced features         (resume)\n  _target_tf_{split}.pkl         partial target teacher-forced features        (resume)\n  traces_{split}.npz             final traces (``traces.save_traces``)\n  timing.json                    t_draft, t_target, c, verify(g)               (skipped if present)\n  run_info_{split}.json\n"""\nfrom __future__ import annotations\n\nimport gc\nimport json\nimport os\nimport pickle\nimport platform\nimport shutil\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .traces import TOPK, Trace, save_traces, summary\nfrom .vocab import VocabMasks, build_masks\n\nPROMPT_TEMPLATE = "Translate the following English sentence into Hindi. Output only the Hindi translation.\\n\\n{s}"\n\n\n# --------------------------------------------------------------------------- #\n# loading\n# --------------------------------------------------------------------------- #\n\ndef _torch():\n    import torch\n    return torch\n\n\ndef sync_all() -> None:\n    torch = _torch()\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(i)\n\n\ndef gpu_report(tag: str = "") -> None:\n    torch = _torch()\n    if torch.cuda.is_available():\n        print(tag, " | ".join(f"GPU{i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB allocated"\n                              for i in range(torch.cuda.device_count())))\n\n\ndef from_pretrained(name: str, dtype, **kw):\n    """``AutoModelForCausalLM.from_pretrained`` across transformers versions (dtype vs torch_dtype)."""\n    from transformers import AutoModelForCausalLM\n    try:\n        return AutoModelForCausalLM.from_pretrained(name, dtype=dtype, **kw).eval()\n    except TypeError:\n        return AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, **kw).eval()\n\n\ndef load_tokenizer(name: str):\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(name)\n    tok.padding_side = "left"\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\n\ndef load_target(name: str, dtype, max_memory: dict | None = None, device_map="auto"):\n    """Target model, layer-split over the available GPUs (``device_map=\'auto\'``) or on CPU."""\n    torch = _torch()\n    if torch.cuda.is_available():\n        m = from_pretrained(name, dtype, device_map=device_map, max_memory=max_memory,\n                            low_cpu_mem_usage=True)\n        print("modules per device:", dict(Counter(m.hf_device_map.values())))\n    else:\n        m = from_pretrained(name, dtype)\n    return m\n\n\ndef load_draft(name: str, dtype, device: str):\n    return from_pretrained(name, dtype).to(device)\n\n\ndef get_eos_ids(model, tok) -> list[int]:\n    """Stop ids: generation-config eos plus ``<|eot_id|>`` (Llama-3 instruct end of turn)."""\n    eos = model.generation_config.eos_token_id\n    ids = set(eos if isinstance(eos, list) else [eos])\n    eot = tok.convert_tokens_to_ids("<|eot_id|>")\n    if eot is not None and eot != tok.unk_token_id:\n        ids.add(eot)\n    return sorted(int(i) for i in ids if i is not None)\n\n\ndef check_draft_target_compat(draft, target, tok_d, tok_t, probe: str) -> None:\n    """Draft and target must share a vocabulary (legacy asserts, kept)."""\n    assert draft.config.vocab_size == target.config.vocab_size, "vocab sizes differ"\n    probe = probe + " नमस्ते दुनिया. Hello world!"\n    assert (tok_d(probe, add_special_tokens=False).input_ids\n            == tok_t(probe, add_special_tokens=False).input_ids), "tokenizers disagree"\n    print("draft/target tokenizers match")\n\n\n# --------------------------------------------------------------------------- #\n# data + prompts\n# --------------------------------------------------------------------------- #\n\ndef load_flores_pairs(split: str, n: int | None = None):\n    """FLORES+ eng_Latn -> hin_Deva rows ``(id, src, ref)`` for ``split`` in {dev, devtest}."""\n    from datasets import load_dataset\n    assert split in ("dev", "devtest"), split\n    en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split=split)\n    hi = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split=split)\n    hi_by_id = {r["id"]: r["text"] for r in hi}\n    rows = [(int(r["id"]), r["text"], hi_by_id[r["id"]]) for r in en]\n    return rows[:n] if n else rows\n\n\ndef build_prompt(tok, s: str) -> str:\n    msgs = [{"role": "user", "content": PROMPT_TEMPLATE.format(s=s)}]\n    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n\ndef read_jsonl(path) -> list[dict]:\n    path = Path(path)\n    if not path.exists():\n        return []\n    with open(path, encoding="utf-8") as fh:\n        return [json.loads(l) for l in fh if l.strip()]\n\n\n# --------------------------------------------------------------------------- #\n# 1. batched greedy target generation (resumable)\n# --------------------------------------------------------------------------- #\n\ndef generate_targets(target, tok, rows, out_jsonl, batch: int, max_new: int, eos_ids,\n                     device: str, progress: bool = True) -> list[dict]:\n    """Greedy, batched (left padding) target outputs, appended to ``out_jsonl`` after every batch.\n\n    Resumes if ``out_jsonl`` holds a prefix of ``rows`` (by id); otherwise starts over.\n    """\n    torch = _torch()\n    from tqdm.auto import tqdm\n    out_jsonl = Path(out_jsonl)\n    ids = [r[0] for r in rows]\n    prompts = [build_prompt(tok, r[1]) for r in rows]\n    records = read_jsonl(out_jsonl)\n    if [r["id"] for r in records] != ids[:len(records)]:\n        print("existing output does not match this run; starting over")\n        records = []\n    print(f"resuming: {len(records)} / {len(rows)} sentences already done")\n    eos_set = set(eos_ids)\n    with open(out_jsonl, "a" if records else "w", encoding="utf-8") as f:\n        rng = range(len(records), len(prompts), batch)\n        for i in (tqdm(rng, desc="generate") if progress else rng):\n            enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True,\n                      add_special_tokens=False).to(device)\n            with torch.no_grad():\n                out = target.generate(**enc, do_sample=False, temperature=None, top_p=None,\n                                      max_new_tokens=max_new, eos_token_id=list(eos_set),\n                                      pad_token_id=tok.pad_token_id)\n            gen = out[:, enc.input_ids.shape[1]:].tolist()\n            for j, g in enumerate(gen):\n                cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))  # keep eos\n                g = g[:cut]\n                pid = enc.input_ids[j][enc.attention_mask[j].bool()].tolist()      # unpadded prompt\n                rid, src, ref = rows[i + j]\n                rec = dict(id=rid, src=src, ref=ref, prompt_ids=pid, out_ids=g,\n                           out_text=tok.decode(g, skip_special_tokens=True).strip())\n                records.append(rec)\n                f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            f.flush()\n    n_hit = sum(len(r["out_ids"]) >= max_new for r in records)\n    print(f"{len(records)} target outputs in {out_jsonl.name}; {n_hit} hit MAX_NEW={max_new}")\n    return records\n\n\ndef generate_unbatched(model, tok, prompt_ids: list[int], max_new: int, eos_ids, device: str) -> list[int]:\n    """Plain greedy generation for one prompt, no padding (reference for losslessness, B9)."""\n    torch = _torch()\n    x = torch.tensor([prompt_ids], device=device)\n    with torch.no_grad():\n        out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), do_sample=False,\n                             temperature=None, top_p=None, max_new_tokens=max_new,\n                             eos_token_id=list(eos_ids), pad_token_id=tok.pad_token_id)\n    g = out[0, len(prompt_ids):].tolist()\n    eos_set = set(eos_ids)\n    cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))\n    return g[:cut]\n\n\n# --------------------------------------------------------------------------- #\n# 2./3. teacher-forced passes\n# --------------------------------------------------------------------------- #\n\ndef _mask_matrix(masks: VocabMasks, vocab_size: int, device):\n    """(V, 5) float32 matrix: columns space, punct, eos, mark, cont."""\n    torch = _torch()\n    cols = [masks.padded("starts_space", vocab_size), masks.padded("starts_punct", vocab_size),\n            np.concatenate([masks.is_eos, np.zeros(vocab_size - masks.size, bool)]),\n            masks.padded("starts_mark", vocab_size), masks.padded("starts_cont", vocab_size)]\n    return torch.tensor(np.stack(cols, 1).astype(np.float32), device=device)\n\n\ndef _logits_for_output(model, prompt_ids, out_ids, device):\n    """Logits (float32) predicting each output token: row t is q_t (conditioned on prompt + y[:t])."""\n    torch = _torch()\n    x = torch.tensor([list(prompt_ids) + list(out_ids)], device=device)\n    with torch.no_grad():\n        lg = model(x).logits[0, len(prompt_ids) - 1:-1]\n    return lg.float()\n\n\ndef draft_features(draft, rec: dict, M, device: str, topk: int = TOPK) -> dict[str, np.ndarray]:\n    """Per-position draft features for one record (all float32 on host)."""\n    torch = _torch()\n    lg = _logits_for_output(draft, rec["prompt_ids"], rec["out_ids"], device)\n    lp = torch.log_softmax(lg, -1)\n    p = lp.exp()\n    H = -(p * lp).sum(-1)\n    t2 = p.topk(2, -1).values\n    tk = lp.topk(topk, -1)\n    y = torch.tensor(rec["out_ids"], device=lg.device)\n    P = p @ M.to(lg.device)                       # (n, 5): space, punct, eos, mark, cont\n    f = dict(draft_argmax=lp.argmax(-1), H=H, top1=t2[:, 0], top2=t2[:, 1],\n             p_target=p.gather(1, y[:, None]).squeeze(1),\n             p_space=P[:, 0], p_punct=P[:, 1], p_eos=P[:, 2], p_newword=P[:, 0] + P[:, 1] + P[:, 2],\n             p_mark=P[:, 3], p_cont=P[:, 4], draft_topk_ids=tk.indices, draft_topk_logp=tk.values)\n    return {k: v.cpu().numpy() for k, v in f.items()}\n\n\ndef target_features(target, rec: dict, draft_argmax: np.ndarray, device: str,\n                    topk: int = TOPK) -> dict[str, np.ndarray]:\n    torch = _torch()\n    lg = _logits_for_output(target, rec["prompt_ids"], rec["out_ids"], device)\n    lp = torch.log_softmax(lg, -1)\n    tk = lp.topk(topk, -1)\n    da = torch.tensor(np.asarray(draft_argmax), device=lg.device, dtype=torch.long)\n    f = dict(target_topk_ids=tk.indices, target_topk_logp=tk.values,\n             target_logp_of_draft_argmax=lp.gather(1, da[:, None]).squeeze(1))\n    return {k: v.cpu().numpy() for k, v in f.items()}\n\n\ndef _resumable_pass(records, fn, partial_path, desc: str, save_every: int = 50, progress=True):\n    """Run ``fn(rec)`` for every record, pickling partial results so a dead session can resume."""\n    from tqdm.auto import tqdm\n    partial_path = Path(partial_path)\n    done: dict = {}\n    if partial_path.exists():\n        with open(partial_path, "rb") as fh:\n            done = pickle.load(fh)\n        print(f"{desc}: resuming with {len(done)} / {len(records)} done")\n    it = tqdm(records, desc=desc) if progress else records\n    since = 0\n    for rec in it:\n        if rec["id"] in done:\n            continue\n        done[rec["id"]] = fn(rec)\n        since += 1\n        if since >= save_every:\n            _atomic_pickle(done, partial_path)\n            since = 0\n    _atomic_pickle(done, partial_path)\n    return done\n\n\ndef _atomic_pickle(obj, path: Path) -> None:\n    tmp = path.with_suffix(".tmp")\n    with open(tmp, "wb") as fh:\n        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)\n    os.replace(tmp, path)\n\n\ndef run_draft_pass(draft, records, masks: VocabMasks, out_dir, split: str, device: str, progress=True):\n    vocab = draft.get_output_embeddings().weight.shape[0]\n    M = _mask_matrix(masks, vocab, device)\n    return _resumable_pass(records, lambda r: draft_features(draft, r, M, device),\n                           Path(out_dir) / f"_draft_tf_{split}.pkl", "draft teacher-forced", progress=progress)\n\n\ndef run_target_pass(target, records, draft_feats: dict, out_dir, split: str, device: str, progress=True):\n    return _resumable_pass(records, lambda r: target_features(target, r, draft_feats[r["id"]]["draft_argmax"], device),\n                           Path(out_dir) / f"_target_tf_{split}.pkl", "target teacher-forced", progress=progress)\n\n\ndef assemble_traces(records, draft_feats: dict, target_feats: dict) -> list[Trace]:\n    out = []\n    for r in records:\n        d, t = draft_feats[r["id"]], target_feats[r["id"]]\n        out.append(Trace(id=int(r["id"]), n=len(r["out_ids"]), target_ids=np.asarray(r["out_ids"], np.int32),\n                         **d, **t))\n    return out\n\n\ndef target_argmax_check(traces: list[Trace]) -> dict:\n    """Rate at which the unpadded teacher-forced target argmax equals the generated y[t]."""\n    agree, diffs, total = 0, [], 0\n    for t in traces:\n        am = t.target_topk_ids[:, 0]\n        eq = am == t.target_ids\n        agree += int(eq.sum())\n        total += t.n\n        for pos in np.flatnonzero(~eq):\n            lp = t.target_topk_logp[pos]\n            diffs.append(dict(id=t.id, pos=int(pos), y=int(t.target_ids[pos]), argmax=int(am[pos]),\n                              margin=float(lp[0] - lp[1])))\n    return dict(rate=agree / max(total, 1), positions=total, n_diff=len(diffs), diffs=diffs)\n\n\n# --------------------------------------------------------------------------- #\n# 4. timing\n# --------------------------------------------------------------------------- #\n\ndef time_fwd(model, q: int = 1, ctx: int = 150, steps: int = 60, reps: int = 3, device: str = "cuda:0") -> float:\n    """Median latency (s) of a forward with ``q`` new tokens on top of a KV cache of ``ctx`` tokens."""\n    torch = _torch()\n    ts = []\n    with torch.no_grad():\n        for _ in range(reps):\n            x = torch.randint(1000, 20000, (1, ctx), device=device)\n            o = model(x, use_cache=True)\n            pkv = o.past_key_values\n            for s in range(steps + 10):\n                inp = torch.randint(1000, 20000, (1, q), device=device)\n                sync_all()\n                t = time.perf_counter()\n                o = model(inp, past_key_values=pkv, use_cache=True)\n                sync_all()\n                dt = time.perf_counter() - t\n                pkv = o.past_key_values\n                if s >= 10:\n                    ts.append(dt)\n    return float(np.median(ts))\n\n\ndef run_timing(draft, target, out_path, device: str, setup: dict, steps: int = 60, reps: int = 3,\n               max_g: int = 8) -> dict:\n    """Legacy timing (c and verify(g), g=1..max_g) plus draft q=1 at ctx 150 and 400."""\n    out_path = Path(out_path)\n    if out_path.exists():\n        print(f"{out_path.name} exists, timing skipped")\n        return json.loads(out_path.read_text())\n    t_d = time_fwd(draft, 1, steps=steps, reps=reps, device=device)\n    t_t = time_fwd(target, 1, steps=steps, reps=reps, device=device)\n    print(f"draft decode step: {t_d * 1000:.2f} ms | target decode step: {t_t * 1000:.2f} ms | c = {t_d / t_t:.3f}")\n    ver = {g: time_fwd(target, g + 1, steps=steps, reps=reps, device=device) for g in range(1, max_g + 1)}\n    print({g: f"{v * 1000:.1f} ms" for g, v in ver.items()})\n    extra = {str(ctx): time_fwd(draft, 1, ctx=ctx, steps=steps, reps=reps, device=device) for ctx in (150, 400)}\n    T = dict(t_draft=t_d, t_target=t_t, c=t_d / t_t, t_target_verify={str(g): v for g, v in ver.items()},\n             t_draft_by_ctx=extra, setup=setup)\n    out_path.write_text(json.dumps(T, indent=1))\n    return T\n\n\n# --------------------------------------------------------------------------- #\n# run info + zip\n# --------------------------------------------------------------------------- #\n\ndef run_info(git_hash: str, **extra) -> dict:\n    torch = _torch()\n    import transformers\n    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []\n    return dict(git_hash=git_hash, torch=torch.__version__, transformers=transformers.__version__,\n                python=platform.python_version(), gpus=gpus, time=time.strftime("%Y-%m-%d %H:%M:%S"), **extra)\n\n\ndef zip_dir(src_dir, zip_path, include=None) -> str:\n    """Zip files of ``src_dir`` (only names in ``include`` if given; flat, no folder) to ``zip_path``.\n    Never deletes or moves anything in ``src_dir``. Returns the zip path."""\n    import zipfile\n    src_dir, zip_path = Path(src_dir), Path(zip_path)\n    names = list(include) if include is not None else sorted(p.name for p in src_dir.iterdir() if p.is_file())\n    missing = [n for n in names if not (src_dir / n).exists()]\n    if missing:\n        raise FileNotFoundError(f"cannot zip, missing in {src_dir}: {missing}")\n    tmp = zip_path.with_name(zip_path.name + ".part")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        for n in names:\n            z.write(src_dir / n, arcname=n)\n    os.replace(tmp, zip_path)\n    return str(zip_path)\n\n\ndef free(*objs) -> None:\n    for o in objs:\n        del o\n    gc.collect()\n    torch = _torch()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\n# --------------------------------------------------------------------------- #\n# full pipeline for one split (used by notebook 01 and the smoke test)\n# --------------------------------------------------------------------------- #\n\ndef run_split(*, split: str, out_dir, target, draft, tok, tok_d, n: int | None, batch: int,\n              max_new: int, device: str, git_hash: str, timing_kw: dict | None = None,\n              setup_extra: dict | None = None, rows=None, progress: bool = True,\n              zip_name: str | None = None) -> dict:\n    """Trace collection for one split. Returns a dict with paths and summaries.\n\n    timing_kw: passed to ``run_timing`` (steps, reps, max_g); setup_extra: added to timing.json setup\n    (e.g. max_memory, GPU names).\n    """\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rows = rows if rows is not None else load_flores_pairs(split, n)\n    eos_ids = get_eos_ids(target, tok)\n    print(f"{split}: {len(rows)} sentences; eos ids {eos_ids}")\n\n    records = generate_targets(target, tok, rows, out_dir / f"target_outputs_{split}.jsonl",\n                               batch, max_new, eos_ids, device, progress=progress)\n    check_draft_target_compat(draft, target, tok_d, tok, records[0]["out_text"])\n\n    masks = build_masks(tok, eos_ids)\n    dfe = run_draft_pass(draft, records, masks, out_dir, split, device, progress=progress)\n    tfe = run_target_pass(target, records, dfe, out_dir, split, device, progress=progress)\n    traces = assemble_traces(records, dfe, tfe)\n    chk = target_argmax_check(traces)\n    print(f"target teacher-forced argmax == y[t]: {chk[\'rate\']:.4f} ({chk[\'n_diff\']} positions differ)")\n    for d in chk["diffs"][:20]:\n        print("   ", d)\n    if chk["rate"] < 0.995:\n        print("WARNING: target argmax agreement below 0.995")\n\n    import transformers\n    meta = dict(split=split, target=getattr(target.config, "_name_or_path", ""),\n                draft=getattr(draft.config, "_name_or_path", ""), max_new=max_new, batch=batch,\n                eos_ids=eos_ids, git_hash=git_hash, transformers=transformers.__version__,\n                target_argmax_check={k: v for k, v in chk.items() if k != "diffs"},\n                target_argmax_diffs=chk["diffs"][:500])\n    npz = out_dir / f"traces_{split}.npz"\n    save_traces(npz, traces, meta)\n    S = summary(traces)\n    print(f"positions: {S[\'positions\']} | agreement: {S[\'agreement\']:.3f} | mean H: {S[\'mean_H\']:.3f} "\n          f"(match {S[\'mean_H_match\']:.3f}, mismatch {S[\'mean_H_mismatch\']:.3f})")\n\n    setup = dict(target=meta["target"], draft=meta["draft"], dtype=str(next(target.parameters()).dtype),\n                 device=device, **(setup_extra or {}))\n    T = run_timing(draft, target, out_dir / "timing.json", device, setup, **(timing_kw or {}))\n\n    info = run_info(git_hash, split=split, n_sentences=len(rows), summary=S,\n                    target=meta["target"], draft=meta["draft"])\n    (out_dir / f"run_info_{split}.json").write_text(json.dumps(info, indent=1, ensure_ascii=False))\n    zip_name = zip_name or f"traces_v2_{split}"\n    zp = zip_dir(out_dir, out_dir.parent / f"{zip_name}.zip",\n                 include=[npz.name, f"target_outputs_{split}.jsonl", "timing.json", f"run_info_{split}.json"])\n    return dict(zip=zp, npz=str(npz), summary=S, timing=T, target_check=chk)\n\n\n# --------------------------------------------------------------------------- #\n# notebook 01 entry point (also called by scripts/smoke_cpu.py)\n# --------------------------------------------------------------------------- #\n\nDEFAULT_CONFIG_01 = dict(\n    SPLIT="dev",                                    # "dev" or "devtest": one split per run\n    N=None,                                         # None = all sentences of the split\n    BATCH=16, MAX_NEW=256,\n    SMOKE=False,                                    # True: CPU, fp32, one model as target+draft\n    TARGET="meta-llama/Meta-Llama-3-8B-Instruct",\n    DRAFT="meta-llama/Llama-3.2-1B-Instruct",\n    MAX_MEMORY={0: "8GiB", 1: "10GiB"},\n    OUT_ROOT="/kaggle/working",\n    GIT_HASH="unknown",\n    TIMING=dict(steps=60, reps=3, max_g=8),\n    RESUME_FROM=None,   # folder with partial outputs of an earlier (dead) run, e.g. /kaggle/input/<...>\n)\n\n\ndef _prepare_out_dir(out_dir: Path, out_root: Path, resume_from) -> None:\n    """Copy partial outputs from ``resume_from`` (searched recursively) and reuse a timing.json\n    measured by another split\'s run in this session."""\n    out_dir.mkdir(parents=True, exist_ok=True)\n    if resume_from:\n        src = Path(resume_from)\n        for f in src.rglob("*"):\n            if f.is_file() and f.parent.name == out_dir.name and not (out_dir / f.name).exists():\n                shutil.copy2(f, out_dir / f.name)\n                print("resume: copied", f)\n    if not (out_dir / "timing.json").exists():\n        for other in sorted(out_root.glob("traces_v2_*/timing.json")):\n            shutil.copy2(other, out_dir / "timing.json")\n            print("reusing timing from", other)\n            break\n\n\ndef notebook01_main(cfg: dict) -> dict:\n    """Run trace collection for ``cfg[\'SPLIT\']``. Writes ``{OUT_ROOT}/traces_v2_{split}/`` and\n    ``{OUT_ROOT}/traces_v2_{split}.zip``. With ``SMOKE=True`` the draft model is used as both\n    target and draft (fp32, CPU, loaded once) and only 3 sentences are processed."""\n    torch = _torch()\n    cfg = {**DEFAULT_CONFIG_01, **cfg}\n    split = cfg["SPLIT"]\n    out_dir = Path(cfg["OUT_ROOT"]) / f"traces_v2_{split}"\n    _prepare_out_dir(out_dir, Path(cfg["OUT_ROOT"]), cfg["RESUME_FROM"])\n    if cfg["SMOKE"]:\n        device, dtype = "cpu", torch.float32\n        tok = load_tokenizer(cfg["DRAFT"])\n        draft = target = load_draft(cfg["DRAFT"], dtype, device)\n        tok_d = tok\n        n = cfg["N"] or 3\n        setup_extra = dict(smoke=True)\n    else:\n        assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"\n        for i in range(torch.cuda.device_count()):\n            p = torch.cuda.get_device_properties(i)\n            print(f"GPU{i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")\n        device, dtype = "cuda:0", torch.float16\n        tok = load_tokenizer(cfg["TARGET"])\n        tok_d = load_tokenizer(cfg["DRAFT"])\n        target = load_target(cfg["TARGET"], dtype, max_memory=cfg["MAX_MEMORY"])\n        gpu_report("after loading target:")\n        draft = load_draft(cfg["DRAFT"], dtype, device)\n        gpu_report("after loading draft:")\n        n = cfg["N"]\n        setup_extra = dict(max_memory={str(k): v for k, v in cfg["MAX_MEMORY"].items()},\n                           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])\n    res = run_split(split=split, out_dir=out_dir, target=target, draft=draft, tok=tok, tok_d=tok_d,\n                    n=n, batch=cfg["BATCH"], max_new=cfg["MAX_NEW"], device=device,\n                    git_hash=cfg["GIT_HASH"], timing_kw=cfg["TIMING"], setup_extra=setup_extra)\n    print("\\n" + "=" * 70)\n    print(f"DONE. Download  {Path(res[\'zip\']).name}  from the notebook\'s Output tab")\n    print(f"and unzip it into  data/traces_v2/  in the repo (files land directly in that folder).")\n    print("=" * 70)\n    return res\n',
    'wordheads': '"""Word-completion heads on the draft model.\n\nPosition convention (same as the Phase-1 traces): for an output ``y[0..n-1]`` after prompt ``P``,\n``h_t`` is the draft\'s last (post-norm) hidden state at input index ``len(P) + t - 1``; the frozen LM\nhead maps it to the draft distribution over ``y[t]`` (row t of a trace).\n\n* head k (k = 1..K) on ``h_t`` predicts ``y[t+k]``: Medusa-1 residual block ``z = h + SiLU(W_k h)``\n  (``W_k`` zero-initialised) followed by the frozen, shared LM head;\n* the continuation head ``Linear(d, K)`` predicts ``c_k[t] = 1`` iff ``y[t+1..t+k]`` all continue the\n  word of ``y[t]`` (none of them is word-initial).\n\nVariants trained side by side on the same draft forward passes (same data, same order):\n  v0  plain MTP:     sum_k w_k CE_k over all positions\n  v1  word-guided:   CE_k only where c_k = 1, plus LAMBDA_CONT * BCE(continuation head)\n  v2  adaptive:      v1\'s K+1 losses with uncertainty weighting  sum_i exp(-s_i) L_i + s_i\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nVARIANTS = ("v0", "v1", "v2")\n\nDEFAULT_WH = dict(\n    DRAFT=["meta-llama/Llama-3.2-1B-Instruct", "unsloth/Llama-3.2-1B-Instruct"],\n    K=3, HEAD_W=0.8, LAMBDA_CONT=0.5,\n    TRAIN_DATASET="cfilt/iitb-english-hindi", N_TRAIN=60000, N_VAL=500, SEED=0,\n    MIN_HI_CHARS=15, MAX_HI_CHARS=400, MIN_WORDS=6, MAX_LEN=320,\n    BATCH=32, LR=1e-3, WD=0.0, WARMUP=100, POS_PER_HEAD=768, EVAL_EVERY=300, MAX_STEPS=None,\n    DUMP_TOPK=8, TIME_Q=[1, 2, 3, 4, 5, 6, 8], TIME_CTX=150, TIME_STEPS=40, TIME_REPS=3,\n    OUT_ROOT="/kaggle/working", SEARCH=["/kaggle/input"], SMOKE=False,\n)\n\n\ndef _torch():\n    import torch\n    return torch\n\n\n# --------------------------------------------------------------------------- #\n# labels\n# --------------------------------------------------------------------------- #\n\ndef word_initial_flags(tok, ids) -> np.ndarray:\n    """Gold \'token starts a new word\' flags of an output sequence (same labeller as Phase 1)."""\n    from .boundaries import token_labels\n    from .vocab import special_ids, token_bytes_list\n    spec = special_ids(tok)\n    tb = token_bytes_list(tok, ids)\n    return np.asarray(token_labels(tb, special=[int(i) in spec for i in ids])["word_initial"], bool)\n\n\ndef continuation_targets(word_initial: np.ndarray, K: int) -> np.ndarray:\n    """c[t, k-1] = 1 iff y[t+1..t+k] exist and none is word-initial (all continue y[t]\'s word)."""\n    n = len(word_initial)\n    c = np.zeros((n, K), bool)\n    for t in range(n):\n        ok = True\n        for k in range(1, K + 1):\n            ok = ok and (t + k < n) and not word_initial[t + k]\n            c[t, k - 1] = ok\n    return c\n\n\ndef continuation_vocab(tok) -> np.ndarray:\n    """Ids of tokens that can only continue a Devanagari word: non-empty, no leading space, and every\n    byte decodes to Devanagari (U+0900-U+097F), ZWJ/ZWNJ, or is part of an incomplete UTF-8 sequence\n    of such a character. Inside a word these are the only possible next tokens, so a word-completion\n    head can score this small vocabulary instead of the full one."""\n    from .vocab import special_ids, token_bytes_list\n    spec = special_ids(tok)\n    n = len(tok)\n    tb = token_bytes_list(tok, list(range(n)))\n    keep = []\n    for i, b in enumerate(tb):\n        if i in spec or not b or b[:1] in (b" ", b"\\n", b"\\t"):\n            continue\n        txt = b.decode("utf-8", errors="ignore")\n        if not txt and all(x >= 0x80 for x in b):        # pure UTF-8 fragment (part of a character)\n            keep.append(i)\n            continue\n        if txt and all("ऀ" <= ch <= "ॿ" or ch in "\u200c\u200d" for ch in txt):\n            keep.append(i)\n    return np.asarray(keep, np.int64)\n\n\n# --------------------------------------------------------------------------- #\n# model\n# --------------------------------------------------------------------------- #\n\ndef make_heads(d: int, K: int):\n    torch = _torch()\n    nn = torch.nn\n\n    class WordHeads(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.K = K\n            self.blocks = nn.ModuleList([nn.Linear(d, d) for _ in range(K)])\n            for b in self.blocks:\n                nn.init.zeros_(b.weight)\n                nn.init.zeros_(b.bias)\n            self.cont = nn.Linear(d, K)\n            self.log_var = nn.Parameter(torch.zeros(K + 1))   # used by v2 only\n\n        def z(self, h, k: int):                     # k = 1..K\n            return h + torch.nn.functional.silu(self.blocks[k - 1](h))\n\n    return WordHeads()\n\n\ndef lm_logits(z, W):\n    """Logits of the frozen LM head (weight W, [V, d], half precision) for hidden rows z (float32)."""\n    return (z.to(W.dtype) @ W.t()).float()\n\n\ndef get_backbone_and_lm_weight(model):\n    base = getattr(model, "model", None) or model.get_decoder()\n    return base, model.get_output_embeddings().weight\n\n\ndef hidden_states(model, input_ids, attention_mask):\n    """Last post-norm hidden states [B, T, d] (what the LM head reads)."""\n    torch = _torch()\n    base, _ = get_backbone_and_lm_weight(model)\n    with torch.no_grad():\n        return base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n\n\n# --------------------------------------------------------------------------- #\n# data\n# --------------------------------------------------------------------------- #\n\ndef encode_pair(tok, src: str, hi: str, eos_id: int) -> tuple[list[int], list[int]]:\n    from .gpu_traces import build_prompt\n    p = tok(build_prompt(tok, src), add_special_tokens=False).input_ids\n    y = tok(hi, add_special_tokens=False).input_ids + [eos_id]\n    return p, y\n\n\ndef make_example(tok, prompt_ids, out_ids, K: int) -> dict:\n    wi = word_initial_flags(tok, out_ids)\n    return dict(prompt_ids=list(prompt_ids), out_ids=list(out_ids),\n                word_initial=wi, cont=continuation_targets(wi, K))\n\n\ndef _deva_frac(s: str) -> float:\n    letters = [ch for ch in s if ch.isalpha()]\n    return sum("ऀ" <= ch <= "ॿ" for ch in letters) / max(1, len(letters))\n\n\n_UI_CHARS = set(r"%_<>/\\{}[]|=@#~^*")\n\n\ndef keep_pair(en: str, hi: str, cfg: dict) -> bool:\n    """Sentence-like pairs only: IITB contains many software-UI strings (GNOME/KDE localisation)."""\n    if not (cfg["MIN_HI_CHARS"] <= len(hi) <= cfg["MAX_HI_CHARS"]):\n        return False\n    if len(en.split()) < cfg["MIN_WORDS"] or len(hi.split()) < cfg["MIN_WORDS"] - 1:\n        return False\n    if en[-1:] not in ".?!\\"\'" or any(ch in _UI_CHARS for ch in en):\n        return False\n    return _deva_frac(hi) >= 0.9\n\n\ndef load_train_pairs(cfg: dict, log=print) -> list[tuple[str, str]]:\n    """N_TRAIN + N_VAL IITB en-hi pairs, sampled uniformly at random (seeded) from the whole train split\n    and filtered with ``keep_pair``."""\n    if cfg.get("TRAIN_PAIRS") is not None:\n        return list(cfg["TRAIN_PAIRS"])\n    from datasets import load_dataset\n    ds = load_dataset(cfg["TRAIN_DATASET"], split="train")\n    need = cfg["N_TRAIN"] + cfg["N_VAL"]\n    order = np.random.default_rng(cfg["SEED"]).permutation(len(ds))\n    out, seen = [], 0\n    for start in range(0, len(order), 20000):\n        rows = ds.select(order[start:start + 20000].tolist())["translation"]\n        for r in rows:\n            seen += 1\n            en, hi = r["en"].strip(), r["hi"].strip()\n            if keep_pair(en, hi, cfg):\n                out.append((en, hi))\n                if len(out) >= need:\n                    break\n        if len(out) >= need:\n            break\n    log(f"train pairs: {len(out)} kept of {seen} sampled (split size {len(ds)})")\n    return out\n\n\ndef collate(batch: list[dict], pad_id: int, device):\n    """Right-padded inputs + index tensors of the output rows h_t (t = 0..n-1) for every example."""\n    torch = _torch()\n    L = max(len(b["prompt_ids"]) + len(b["out_ids"]) for b in batch)\n    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)\n    am = torch.zeros((len(batch), L), dtype=torch.long)\n    rows_b, rows_i, tgt, tvalid, cont = [], [], [], [], []\n    for bi, b in enumerate(batch):\n        seq = b["prompt_ids"] + b["out_ids"]\n        ids[bi, :len(seq)] = torch.tensor(seq)\n        am[bi, :len(seq)] = 1\n        P, n = len(b["prompt_ids"]), len(b["out_ids"])\n        y = np.asarray(b["out_ids"], np.int64)\n        K = b["cont"].shape[1]\n        for t in range(n):\n            rows_b.append(bi)\n            rows_i.append(P + t - 1)\n            tk = np.array([y[t + k] if t + k < n else -100 for k in range(1, K + 1)], np.int64)\n            tgt.append(tk)\n            tvalid.append(tk >= 0)\n            cont.append(b["cont"][t])\n    T = lambda a, dt: torch.tensor(np.asarray(a), dtype=dt, device=device)\n    return dict(ids=ids.to(device), am=am.to(device), rows_b=T(rows_b, torch.long), rows_i=T(rows_i, torch.long),\n                tgt=T(tgt, torch.long), tvalid=T(tvalid, torch.bool), cont=T(cont, torch.bool))\n\n\n# --------------------------------------------------------------------------- #\n# losses\n# --------------------------------------------------------------------------- #\n\ndef variant_losses(name: str, heads, h, B, W, K: int, head_w: float, lam_cont: float,\n                   pos_per_head: int, gen) -> tuple:\n    """Total loss and per-part float values for one variant on hidden rows h [N, d] (float32)."""\n    torch = _torch()\n    F = torch.nn.functional\n    parts, logs = [], {}\n    for k in range(1, K + 1):\n        valid = B["tvalid"][:, k - 1]\n        if name in ("v1", "v2"):\n            valid = valid & B["cont"][:, k - 1]\n        idx = valid.nonzero(as_tuple=True)[0]\n        if len(idx) == 0:\n            parts.append(h.sum() * 0.0)\n            continue\n        if len(idx) > pos_per_head:\n            idx = idx[torch.randperm(len(idx), generator=gen, device="cpu")[:pos_per_head].to(idx.device)]\n        lg = lm_logits(heads.z(h[idx], k), W)\n        ce = F.cross_entropy(lg, B["tgt"][idx, k - 1])\n        parts.append(ce)\n        logs[f"ce{k}"] = ce.item()\n    if name == "v0":\n        total = sum(head_w ** (k - 1) * parts[k - 1] for k in range(1, K + 1))\n        return total, logs\n    # continuation head on all positions where y[t+1] exists\n    m = B["tvalid"][:, 0]\n    bce = F.binary_cross_entropy_with_logits(heads.cont(h[m]), B["cont"][m].float())\n    logs["bce"] = bce.item()\n    if name == "v1":\n        total = sum(head_w ** (k - 1) * parts[k - 1] for k in range(1, K + 1)) + lam_cont * bce\n    else:\n        allp = parts + [bce]\n        total = sum(torch.exp(-heads.log_var[i]) * allp[i] + heads.log_var[i] for i in range(len(allp)))\n        logs["w"] = [round(float(torch.exp(-s.detach())), 3) for s in heads.log_var]\n    return total, logs\n\n\n# --------------------------------------------------------------------------- #\n# evaluation helpers\n# --------------------------------------------------------------------------- #\n\ndef head_accuracy(heads, h, B, W, K: int) -> dict:\n    """Top-1 accuracy per head on all valid positions and on in-word (c_k = 1) positions."""\n    torch = _torch()\n    out = {}\n    with torch.no_grad():\n        for k in range(1, K + 1):\n            v = B["tvalid"][:, k - 1]\n            if not v.any():\n                continue\n            pred = lm_logits(heads.z(h[v], k), W).argmax(-1)\n            hit = pred == B["tgt"][v, k - 1]\n            cw = B["cont"][v, k - 1]\n            out[f"acc{k}_all"] = float(hit.float().mean())\n            out[f"acc{k}_inword"] = float(hit[cw].float().mean()) if cw.any() else float("nan")\n        m = B["tvalid"][:, 0]\n        if m.any():\n            pc = torch.sigmoid(heads.cont(h[m])) > 0.5\n            out["cont_acc"] = float((pc == B["cont"][m]).float().mean())\n    return out\n\n\n# --------------------------------------------------------------------------- #\n# training\n# --------------------------------------------------------------------------- #\n\ndef train_heads(model, tok, train: list[dict], val: list[dict], cfg: dict, device, log=print) -> tuple:\n    torch = _torch()\n    K = cfg["K"]\n    base, W = get_backbone_and_lm_weight(model)\n    W = W.detach()\n    d = W.shape[1]\n    torch.manual_seed(cfg["SEED"])\n    heads = {v: make_heads(d, K).to(device) for v in VARIANTS}\n    opt = {v: torch.optim.AdamW(heads[v].parameters(), lr=cfg["LR"], weight_decay=cfg["WD"]) for v in VARIANTS}\n    order = sorted(range(len(train)), key=lambda i: len(train[i]["prompt_ids"]) + len(train[i]["out_ids"]))\n    batches = [order[i:i + cfg["BATCH"]] for i in range(0, len(order), cfg["BATCH"])]\n    random.Random(cfg["SEED"]).shuffle(batches)\n    total = len(batches) if not cfg.get("MAX_STEPS") else min(len(batches), cfg["MAX_STEPS"])\n    sched = {v: torch.optim.lr_scheduler.LambdaLR(\n        opt[v], lambda s: min(1.0, (s + 1) / cfg["WARMUP"]) * max(0.05, 0.5 * (1 + math.cos(math.pi * s / total))))\n        for v in VARIANTS}\n    gen = torch.Generator().manual_seed(cfg["SEED"])\n    pad = tok.pad_token_id if tok.pad_token_id is not None else 0\n    hist, t0 = [], time.time()\n    val_b = [collate(val[i:i + cfg["BATCH"]], pad, device) for i in range(0, len(val), cfg["BATCH"])]\n\n    def evaluate(step):\n        acc = {v: [] for v in VARIANTS}\n        for B in val_b:\n            h = hidden_states(model, B["ids"], B["am"])[B["rows_b"], B["rows_i"]].float()\n            for v in VARIANTS:\n                acc[v].append(head_accuracy(heads[v], h, B, W, K))\n        res = {v: {k: float(np.nanmean([a[k] for a in acc[v] if k in a])) for k in acc[v][0]} for v in VARIANTS}\n        hist.append(dict(step=step, val=res, t=round(time.time() - t0, 1)))\n        log(f"[val step {step}] " + " | ".join(\n            f"{v}: in-word acc " + "/".join(f"{res[v].get(f\'acc{k}_inword\', float(\'nan\')):.3f}" for k in range(1, K + 1))\n            for v in VARIANTS))\n        return res\n\n    for step, bidx in enumerate(batches[:total]):\n        B = collate([train[i] for i in bidx], pad, device)\n        h = hidden_states(model, B["ids"], B["am"])[B["rows_b"], B["rows_i"]].float()\n        logs = {}\n        for v in VARIANTS:\n            loss, lg = variant_losses(v, heads[v], h, B, W, K, cfg["HEAD_W"], cfg["LAMBDA_CONT"],\n                                      cfg["POS_PER_HEAD"], gen)\n            opt[v].zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(heads[v].parameters(), 1.0)\n            opt[v].step()\n            sched[v].step()\n            logs[v] = lg\n        if step % 50 == 0:\n            log(f"step {step}/{total} {time.time() - t0:.0f}s " + json.dumps(\n                {v: {k: (round(x, 3) if isinstance(x, float) else x) for k, x in logs[v].items()} for v in VARIANTS}))\n        if cfg["EVAL_EVERY"] and step % cfg["EVAL_EVERY"] == 0 and step:\n            evaluate(step)\n    final = evaluate(total)\n    return heads, dict(history=hist, final_val=final, steps=total, seconds=round(time.time() - t0, 1))\n\n\n# --------------------------------------------------------------------------- #\n# dump along the target\'s outputs (for the exact trace simulation)\n# --------------------------------------------------------------------------- #\n\ndef dump_predictions(model, tok, heads: dict, records: list[dict], cfg: dict, device) -> dict:\n    """Per output position t of every record: the draft argmax for y[t] (alignment check against the\n    Phase-1 traces), and per variant the heads\' top-k ids / log-probs for y[t+1..t+K] and the\n    continuation-head probabilities. Flat arrays + offsets, like the Phase-1 traces."""\n    torch = _torch()\n    K, TK = cfg["K"], cfg["DUMP_TOPK"]\n    _, W = get_backbone_and_lm_weight(model)\n    W = W.detach()\n    cv = torch.tensor(continuation_vocab(tok), device=W.device)\n    Wcv = W[cv]\n    out = {"ids": [], "lens": [], "draft_argmax": []}\n    for v in VARIANTS:\n        out[f"{v}_ids"], out[f"{v}_logp"], out[f"{v}_cont"] = [], [], []\n        out[f"{v}_cv_id"], out[f"{v}_cv_logp"] = [], []\n    with torch.no_grad():\n        for r in records:\n            P, y = list(r["prompt_ids"]), list(r["out_ids"])\n            x = torch.tensor([P + y], device=device)\n            h = hidden_states(model, x, torch.ones_like(x))[0, len(P) - 1:len(P) - 1 + len(y)].float()\n            out["ids"].append(int(r["id"]))\n            out["lens"].append(len(y))\n            out["draft_argmax"].append(lm_logits(h, W).argmax(-1).cpu().numpy().astype(np.int32))\n            for v in VARIANTS:\n                ids_k, lp_k, cid_k, clp_k = [], [], [], []\n                for k in range(1, K + 1):\n                    z = heads[v].z(h, k)\n                    lp = torch.log_softmax(lm_logits(z, W), -1)\n                    tk = lp.topk(TK, -1)\n                    ids_k.append(tk.indices.cpu().numpy().astype(np.int32))\n                    lp_k.append(tk.values.cpu().numpy().astype(np.float16))\n                    lc = torch.log_softmax(lm_logits(z, Wcv), -1).max(-1)   # restricted to continuation tokens\n                    cid_k.append(cv[lc.indices].cpu().numpy().astype(np.int32))\n                    clp_k.append(lc.values.cpu().numpy().astype(np.float16))\n                out[f"{v}_ids"].append(np.stack(ids_k, 1))      # [n, K, TK]\n                out[f"{v}_logp"].append(np.stack(lp_k, 1))\n                out[f"{v}_cv_id"].append(np.stack(cid_k, 1))    # [n, K]\n                out[f"{v}_cv_logp"].append(np.stack(clp_k, 1))\n                out[f"{v}_cont"].append(torch.sigmoid(heads[v].cont(h)).cpu().numpy().astype(np.float16))\n    lens = np.asarray(out.pop("lens"), np.int64)\n    res = dict(ids=np.asarray(out.pop("ids"), np.int64), offsets=np.concatenate([[0], np.cumsum(lens)]),\n               cont_vocab=cv.cpu().numpy())\n    for k, v in out.items():\n        res[k] = np.concatenate(v, 0)\n    return res\n\n\ndef time_heads(model, tok, heads, cfg: dict, device) -> dict:\n    """Draft latency for q new tokens (KV cache of TIME_CTX) and the heads\' latency (all K heads in one\n    batched matmul; full vocabulary and continuation vocabulary), relative to the draft\'s q = 1 call."""\n    from .gpu_traces import sync_all, time_fwd\n    torch = _torch()\n    tq = {q: time_fwd(model, q=q, ctx=cfg["TIME_CTX"], steps=cfg["TIME_STEPS"], reps=cfg["TIME_REPS"], device=device)\n          for q in cfg["TIME_Q"]}\n    _, W = get_backbone_and_lm_weight(model)\n    Wcv = W[torch.tensor(continuation_vocab(tok), device=W.device)]\n    h = torch.randn(1, W.shape[1], device=device)\n\n    def run(Wm):\n        ts = []\n        with torch.no_grad():\n            for s in range(cfg["TIME_STEPS"] + 10):\n                sync_all()\n                t = time.perf_counter()\n                z = torch.cat([heads.z(h, k) for k in range(1, cfg["K"] + 1)], 0)\n                lm_logits(z, Wm).argmax(-1)\n                torch.sigmoid(heads.cont(h))\n                sync_all()\n                if s >= 10:\n                    ts.append(time.perf_counter() - t)\n        return float(np.median(ts))\n\n    t_full, t_cv = run(W), run(Wcv)\n    return dict(t_draft={str(q): v for q, v in tq.items()},\n                r_q={str(q): v / tq[1] for q, v in tq.items()},\n                t_heads_full=t_full, head_rel_full=t_full / tq[1],\n                t_heads_cv=t_cv, head_rel_cv=t_cv / tq[1], cont_vocab_size=int(Wcv.shape[0]),\n                ctx=cfg["TIME_CTX"])\n\n\n# --------------------------------------------------------------------------- #\n# notebook entry point\n# --------------------------------------------------------------------------- #\n\ndef _find_heads(cfg: dict) -> Path | None:\n    for root in cfg["SEARCH"]:\n        p = Path(root)\n        if p.exists():\n            for f in p.rglob("wordheads_v0.pt"):\n                return f.parent\n    return None\n\n\ndef wordheads_main(cfg_in: dict, target_records: dict) -> dict:\n    """Train v0/v1/v2 heads on the frozen draft, dump predictions along ``target_records``\n    ({split: [records with id, prompt_ids, out_ids]}), time the draft and heads, zip everything."""\n    torch = _torch()\n    from . import gpu_traces as G\n    cfg = dict(DEFAULT_WH)\n    cfg.update(cfg_in)\n    out = Path(cfg["OUT_ROOT"]) / "wordheads"\n    out.mkdir(parents=True, exist_ok=True)\n    logf = open(out / "log.txt", "a", encoding="utf-8")\n\n    def log(msg):\n        print(msg, flush=True)\n        logf.write(msg + "\\n")\n        logf.flush()\n\n    device = "cuda:0" if torch.cuda.is_available() else "cpu"\n    dtype = torch.float16 if device.startswith("cuda") else torch.float32\n    model, name = None, None\n    for nm in cfg["DRAFT"]:\n        try:\n            model = G.from_pretrained(nm, dtype).to(device)\n            name = nm\n            break\n        except Exception as e:  # noqa: BLE001\n            log(f"could not load {nm}: {e}")\n    assert model is not None, "no draft model could be loaded"\n    for p in model.parameters():\n        p.requires_grad_(False)\n    tok = G.load_tokenizer(name)\n    tok.padding_side = "right"\n    eos = G.get_eos_ids(model, tok)\n    eot = tok.convert_tokens_to_ids("<|eot_id|>")\n    eos_id = eot if (eot is not None and eot != tok.unk_token_id and eot in eos) else eos[0]\n    log(f"draft {name} on {device}; eos {eos_id}")\n\n    K = cfg["K"]\n    prev = _find_heads(cfg)\n    info = dict(draft=name, cfg={k: v for k, v in cfg.items() if k != "TRAIN_PAIRS"})\n    if prev is not None:\n        d = get_backbone_and_lm_weight(model)[1].shape[1]\n        heads = {}\n        for v in VARIANTS:\n            heads[v] = make_heads(d, K).to(device)\n            heads[v].load_state_dict({k: t.float() for k, t in torch.load(prev / f"wordheads_{v}.pt").items()})\n        log(f"heads: reused from {prev}")\n        info["train"] = json.loads((prev / "train_info.json").read_text()) if (prev / "train_info.json").exists() else {}\n    else:\n        pairs = load_train_pairs(cfg, log)\n        rnd = random.Random(cfg["SEED"])\n        rnd.shuffle(pairs)\n        ex = []\n        for en, hi in pairs:\n            p, y = encode_pair(tok, en, hi, eos_id)\n            if len(p) + len(y) <= cfg["MAX_LEN"]:\n                ex.append(make_example(tok, p, y, K))\n        val, train = ex[:cfg["N_VAL"]], ex[cfg["N_VAL"]:]\n        wi = np.concatenate([e["word_initial"] for e in train])\n        log(f"examples: train {len(train)} val {len(val)}; output tokens {len(wi)}; continuation share {1 - wi.mean():.3f}")\n        heads, tinfo = train_heads(model, tok, train, val, cfg, device, log)\n        tinfo["continuation_share"] = float(1 - wi.mean())\n        info["train"] = tinfo\n        for v in VARIANTS:\n            torch.save({k: t.half() for k, t in heads[v].state_dict().items()}, out / f"wordheads_{v}.pt")\n        (out / "train_info.json").write_text(json.dumps(tinfo, indent=1))\n    for v in VARIANTS:\n        heads[v].eval()\n\n    info["dump"] = {}\n    for split, recs in target_records.items():\n        res = dump_predictions(model, tok, heads, recs, cfg, device)\n        np.savez_compressed(out / f"heads_{split}.npz", **res)\n        info["dump"][split] = dict(records=len(recs), positions=int(res["offsets"][-1]))\n        log(f"dump {split}: {info[\'dump\'][split]}")\n    info["timing"] = time_heads(model, tok, heads["v1"], cfg, device)\n    log("timing: " + json.dumps(info["timing"]))\n    info["run"] = G.run_info(cfg.get("BUILD", "nobuild"), draft=name)\n    (out / "run_info.json").write_text(json.dumps(info, indent=1, default=str))\n    zp = G.zip_dir(out, Path(cfg["OUT_ROOT"]) / "wordheads.zip")\n    log(f"wrote {zp}")\n    logf.close()\n    return info\n',
}
for _n, _s in LGMTP_SOURCES.items():
    with open(os.path.join(PKG, _n + '.py'), 'w', encoding='utf-8') as _f:
        _f.write(_s)
sys.path.insert(0, os.path.dirname(PKG))
for _m in [m for m in list(sys.modules) if m == 'lgmtp' or m.startswith('lgmtp.')]:
    del sys.modules[_m]
import lgmtp
LGMTP_BUILD = 'git=db73cf78fd9c-dirty src_sha256=3b29b035b0e9'
print('lgmtp build', LGMTP_BUILD)

In [ ]:
import base64, gzip, io, json
_REC_B64 = '...'
TARGET_RECORDS = json.loads(gzip.decompress(base64.b64decode(_REC_B64)))
print({k: len(v) for k, v in TARGET_RECORDS.items()})

## CONFIG

In [ ]:
CONFIG = dict(
    DRAFT=["meta-llama/Llama-3.2-1B-Instruct", "unsloth/Llama-3.2-1B-Instruct"],
    K=3, N_TRAIN=60000, N_VAL=500, BATCH=32, LR=1e-3, POS_PER_HEAD=768, EVAL_EVERY=300,
    LAMBDA_CONT=0.5, HEAD_W=0.8, MAX_LEN=320,
    BUILD='git=db73cf78fd9c-dirty src_sha256=3b29b035b0e9',
)

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in with Kaggle secret HF_TOKEN")
except Exception as e:
    print(f"No HF_TOKEN ({e}); the ungated unsloth copy of Llama-3.2-1B will be used")
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())

In [ ]:
from lgmtp.wordheads import wordheads_main
INFO = wordheads_main(CONFIG, TARGET_RECORDS)

In [ ]:
import json
print(json.dumps(INFO['train'].get('final_val', {}), indent=1))
print(json.dumps(INFO['timing'], indent=1))